**【课程】**：[贷前策略实战项目(提高班)](https://bzavt.xetlk.com/s/4io33W)
**【店铺】**：[东哥讲风控](https://app7hmmvkwr2019.h5.xiaoeknow.com)
**【作者】**：东哥起飞

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def create_score_card_transformer(base_score, base_bad_odds, pdo):
    """
    通过基准坏好比(bad_odds)和PDO创建评分卡转换函数
    使用公式: score = A - B * log(bad_odds)
    
    参数:
    base_score: 基准分数 (当bad_odds=base_bad_odds时的分数)
    base_bad_odds: 基准坏好比 (坏客户概率/好客户概率)
    pdo: Points to Double Odds (坏好比翻倍时分数减少的点数)
    
    返回:
    包含各种转换方法的字典
    """
    
    # 计算参数B和A
    # 当bad_odds翻倍时，分数减少pdo分
    # score2 - score1 = -B * log(2) = -pdo
    B = pdo / np.log(2)
    A = base_score + B * np.log(base_bad_odds)
    
    print(f"计算得到的参数:")
    print(f"  A = {A:.2f}")
    print(f"  B = {B:.2f}")
    print(f"  基准: bad_odds={base_bad_odds}时, score={base_score}")
    print(f"  PDO: 坏好比翻倍时分数减少{pdo}分")
    print(f"  转换公式: score = {A:.2f} - {B:.2f} * log(bad_odds)")
    
    def bad_odds_to_score(bad_odds):
        """将坏好比转换为分数"""
        return A - B * np.log(bad_odds)
    
    def probability_to_score(p_bad):
        """将坏客户概率转换为分数"""
        # p_bad: 坏客户概率
        bad_odds = p_bad / (1 - p_bad)
        return bad_odds_to_score(bad_odds)
    
    def good_probability_to_score(p_good):
        """将好客户概率转换为分数"""
        # p_good: 好客户概率
        p_bad = 1 - p_good
        bad_odds = p_bad / p_good
        return bad_odds_to_score(bad_odds)
    
    def score_to_bad_odds(score):
        """将分数转换回坏好比"""
        return np.exp((A - score) / B)
    
    def score_to_bad_probability(score):
        """将分数转换回坏客户概率"""
        bad_odds = score_to_bad_odds(score)
        return bad_odds / (1 + bad_odds)
    
    def score_to_good_probability(score):
        """将分数转换回好客户概率"""
        bad_odds = score_to_bad_odds(score)
        return 1 / (1 + bad_odds)
    
    def get_parameters():
        """返回转换参数"""
        return {
            'A': A, 
            'B': B, 
            'base_score': base_score, 
            'base_bad_odds': base_bad_odds, 
            'pdo': pdo,
            'formula': f'score = {A:.2f} - {B:.2f} * log(bad_odds)'
        }
    
    def generate_score_table(good_probabilities=None):
        """生成分数对照表"""
        if good_probabilities is None:
            good_probabilities = [0.99, 0.95, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1, 0.05, 0.01]
        
        print(f"\n分数对照表 (基准bad_odds={base_bad_odds}, 基准分数={base_score}, PDO={pdo})")
        print("=" * 80)
        print(f"{'好客户概率':<10} {'坏客户概率':<10} {'坏好比':<12} {'Score':<8} {'与基准分数差':<12}")
        print("-" * 80)
        
        base_score_val = bad_odds_to_score(base_bad_odds)
        
        for p_good in good_probabilities:
            p_bad = 1 - p_good
            bad_odds = p_bad / p_good
            score = good_probability_to_score(p_good)
            score_diff = score - base_score_val
            
            print(f"{p_good:<10.3f} {p_bad:<10.3f} {bad_odds:<12.3f} {score:<8.1f} {score_diff:<12.1f}")
    
    # 返回包含所有方法的字典
    return {
        'bad_odds_to_score': bad_odds_to_score,
        'probability_to_score': probability_to_score,  # 输入坏客户概率
        'good_probability_to_score': good_probability_to_score,  # 输入好客户概率
        'score_to_bad_odds': score_to_bad_odds,
        'score_to_bad_probability': score_to_bad_probability,
        'score_to_good_probability': score_to_good_probability,
        'get_parameters': get_parameters,
        'generate_score_table': generate_score_table
    }

In [2]:
scorer = create_score_card_transformer(600,1/19,50)

计算得到的参数:
  A = 387.60
  B = 72.13
  基准: bad_odds=0.05263157894736842时, score=600
  PDO: 坏好比翻倍时分数减少50分
  转换公式: score = 387.60 - 72.13 * log(bad_odds)


In [4]:
scorer['probability_to_score'](0.09)

554.4971062656399

In [5]:
scorer['good_probability_to_score'](0.91)

554.4971062656399

In [7]:
scorer['score_to_bad_odds'](800)

0.0032894736842105257

In [9]:
scorer['generate_score_table']()


分数对照表 (基准bad_odds=0.05263157894736842, 基准分数=600, PDO=50)
好客户概率      坏客户概率      坏好比          Score    与基准分数差      
--------------------------------------------------------------------------------
0.990      0.010      0.010        719.1    119.1       
0.950      0.050      0.053        600.0    -0.0        
0.900      0.100      0.111        546.1    -53.9       
0.800      0.200      0.250        487.6    -112.4      
0.700      0.300      0.429        448.7    -151.3      
0.600      0.400      0.667        416.9    -183.1      
0.500      0.500      1.000        387.6    -212.4      
0.400      0.600      1.500        358.4    -241.6      
0.300      0.700      2.333        326.5    -273.5      
0.200      0.800      4.000        287.6    -312.4      
0.100      0.900      9.000        229.1    -370.9      
0.050      0.950      19.000       175.2    -424.8      
0.010      0.990      99.000       56.1     -543.9      


## At Last.好课推荐
### 1）100天风控专家

贷前策略实战项目，4大真实贷前场景实战项目，SQL+Python代码实操

内容详情介绍👉[《贷前策略实战项目（提高班）》](https://mp.weixin.qq.com/s/33nsHN3v_Zf9sPkr-yp_iA)，下单链接👉[点这里](https://bzavt.xetlk.com/s/4io33W)

### 2）pandas进阶宝典

Pandas数据分析的各种骚操作，东哥的原创笔记， 500页飞书图文笔记，近30万字，已经完全体。配套完整代码支持下载，永久访问权限。

内容详情介绍👉[《pandas进阶宝典》](https://mp.weixin.qq.com/s/9WbDFamcK2WywagdfN_f9Q)，包括5大核心图文，下单链接👉[点这里](https://app7hmmvkwr2019.h5.xiaoeknow.com/p/course/ecourse/course_2YD5u0x8FzrAIyM8soEuxnTkP9r)
- 《pandas快速入门》
- 《pandas进阶宝典》
- 《pandas实战项目》
- 《pandas进阶题库》
- 《Numpy速查手册》
- 《正则表达式手册》

以上如有不清楚也可加我微信：`Petery_1966` 咨询